# 02. Limpieza y transformación

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

In [2]:
df_bank = pd.read_csv("../data/raw/bank-additional.csv")

excel = pd.ExcelFile("../data/raw/customer-details.xlsx")
hojas = {nombre: pd.read_excel(excel, sheet_name=nombre) for nombre in excel.sheet_names}

In [3]:
df_clientes = pd.concat(hojas.values(), ignore_index=True)
df_clientes.shape

(43170, 7)

In [4]:
df_bank = df_bank.drop(columns="Unnamed: 0")
df_clientes = df_clientes.drop(columns="Unnamed: 0")

print(df_bank.shape, df_clientes.shape)

(43000, 23) (43170, 6)


In [5]:
renombrar_bank = {
    "age": "edad",
    "job": "trabajo",
    "marital": "estado_civil",
    "education": "educacion",
    "default": "impago",
    "housing": "hipoteca",
    "loan": "prestamo",
    "contact": "medio_contacto",
    "duration": "duracion",
    "campaign": "contactos_campana",
    "pdays": "dias_ultimo_contacto",
    "previous": "contactos_previos",
    "poutcome": "resultado_previo",
    "emp.var.rate": "tasa_var_empleo",
    "cons.price.idx": "indice_precios",
    "cons.conf.idx": "indice_confianza",
    "euribor3m": "euribor_3m",
    "nr.employed": "num_empleados",
    "y": "suscrito",
    "date": "fecha_contacto",
    "latitude": "latitud",
    "longitude": "longitud",
    "id_": "id_cliente",
}

renombrar_clientes = {
    "Income": "ingresos",
    "Kidhome": "num_ninos",
    "Teenhome": "num_adolescentes",
    "Dt_Customer": "fecha_alta",
    "NumWebVisitsMonth": "visitas_web_mes",
    "ID": "id_cliente",
}

df_bank = df_bank.rename(columns=renombrar_bank)
df_clientes = df_clientes.rename(columns=renombrar_clientes)

In [6]:
print(df_bank.columns.tolist())
print(df_clientes.columns.tolist())

['edad', 'trabajo', 'estado_civil', 'educacion', 'impago', 'hipoteca', 'prestamo', 'medio_contacto', 'duracion', 'contactos_campana', 'dias_ultimo_contacto', 'contactos_previos', 'resultado_previo', 'tasa_var_empleo', 'indice_precios', 'indice_confianza', 'euribor_3m', 'num_empleados', 'suscrito', 'fecha_contacto', 'latitud', 'longitud', 'id_cliente']
['ingresos', 'num_ninos', 'num_adolescentes', 'fecha_alta', 'visitas_web_mes', 'id_cliente']


In [7]:
columnas_coma = ["indice_precios", "indice_confianza", "euribor_3m", "num_empleados"]

for col in columnas_coma:
    df_bank[col] = pd.to_numeric(df_bank[col].str.replace(",", ".", regex=False))

In [8]:
df_bank[columnas_coma].dtypes

indice_precios      float64
indice_confianza    float64
euribor_3m          float64
num_empleados       float64
dtype: object

In [9]:
df_bank[columnas_coma].isnull().sum()

indice_precios       471
indice_confianza       0
euribor_3m          9256
num_empleados          0
dtype: int64

In [ ]:
# BORRAR 

df_bank.sample(5)


,edad,trabajo,estado_civil,educacion,impago,hipoteca,prestamo,medio_contacto,duracion,contactos_campana,dias_ultimo_contacto,contactos_previos,resultado_previo,tasa_var_empleo,indice_precios,indice_confianza,euribor_3m,num_empleados,suscrito,fecha_contacto,latitud,longitud,id_cliente
19260,45.0,blue-collar,MARRIED,basic.4y,NaN,0.0,0.0,cellular,131,1,999,0,NONEXISTENT,1.4,93.444,-36.1,4.967,5228.1,no,24-julio-2018,37.084,-72.271,ac8bb012-b7cb-40be-ae34-346513cb8213
14511,56.0,management,MARRIED,basic.4y,NaN,1.0,0.0,cellular,61,2,999,0,NONEXISTENT,1.4,93.918,-42.7,NaN,5228.1,no,28-junio-2016,47.577,-79.770,34c1aacc-5778-46f1-9a28-34382e02e34c
20451,54.0,services,MARRIED,basic.4y,0.0,1.0,0.0,cellular,50,4,999,0,NONEXISTENT,1.4,93.444,-36.1,4.966,5228.1,no,19-noviembre-2019,37.880,-91.650,7a59576c-aeac-43de-9abe-54014b0ba089
2009,27.0,blue-collar,MARRIED,basic.9y,NaN,0.0,0.0,telephone,322,1,999,0,NONEXISTENT,1.1,93.994,-36.4,4.857,5191.0,no,6-agosto-2016,27.727,-99.948,ff25e152-6f7f-4aeb-b213-080d875c3792
32352,57.0,services,DIVORCED,NaN,0.0,0.0,0.0,cellular,405,3,999,0,NONEXISTENT,-1.8,92.893,-46.2,1.313,5099.1,no,25-junio-2015,39.511,-87.599,41c37d5e-2945-4d9d-82a2-1be8b7e14871


In [12]:
meses = {
    "enero": "01", "febrero": "02", "marzo": "03", "abril": "04",
    "mayo": "05", "junio": "06", "julio": "07", "agosto": "08",
    "septiembre": "09", "octubre": "10", "noviembre": "11", "diciembre": "12",
}

In [16]:
partes = df_bank["fecha_contacto"].str.split("-", expand=True)
partes.head()

,0,1,2
0,2,agosto,2019
1,14,septiembre,2016
2,15,febrero,2019
3,29,noviembre,2015
4,29,enero,2017


In [17]:
meses_traducidos = partes[1].map(meses)
print("Meses sin traducir:", (meses_traducidos.isnull() & partes[1].notnull()).sum())

Meses sin traducir: 0


In [18]:
fecha_texto = partes[2] + "-" + meses_traducidos + "-" + partes[0]
df_bank["fecha_contacto"] = pd.to_datetime(fecha_texto, format="%Y-%m-%d")

In [19]:
print(df_bank["fecha_contacto"].dtype)
print(df_bank["fecha_contacto"].isnull().sum())
print(df_bank["fecha_contacto"].min(), "->", df_bank["fecha_contacto"].max())

datetime64[us]
248
2015-01-01 00:00:00 -> 2019-12-31 00:00:00


In [22]:
columnas_texto = [col for col in df_bank.select_dtypes(include="str").columns if col != "id_cliente"]

for col in columnas_texto:
    df_bank[col] = df_bank[col].str.lower().str.strip()

In [26]:
print(df_bank["estado_civil"].value_counts(dropna=False))
print('==========')
print(df_bank["resultado_previo"].value_counts(dropna=False))

estado_civil
married     25999
single      12105
divorced     4811
NaN            85
Name: count, dtype: int64
resultado_previo
nonexistent    37103
failure         4461
success         1436
Name: count, dtype: int64


In [27]:
df_bank.info()
df_clientes.info()

<class 'pandas.DataFrame'>
RangeIndex: 43000 entries, 0 to 42999
Data columns (total 23 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   edad                  37880 non-null  float64       
 1   trabajo               42655 non-null  str           
 2   estado_civil          42915 non-null  str           
 3   educacion             41193 non-null  str           
 4   impago                34019 non-null  float64       
 5   hipoteca              41974 non-null  float64       
 6   prestamo              41974 non-null  float64       
 7   medio_contacto        43000 non-null  str           
 8   duracion              43000 non-null  int64         
 9   contactos_campana     43000 non-null  int64         
 10  dias_ultimo_contacto  43000 non-null  int64         
 11  contactos_previos     43000 non-null  int64         
 12  resultado_previo      43000 non-null  str           
 13  tasa_var_empleo       43000

## Resumen de la Sesión 3
- Concatenadas las 3 hojas del Excel (43.170 filas).
- Eliminada la columna `Unnamed: 0` en ambos datasets.
- Renombradas las columnas al español (formato snake_case, sin tildes).
- Convertidas a número 4 columnas con coma decimal.
- Convertida `fecha_contacto` de texto en español a fecha.
- Pasadas a minúsculas las columnas de texto.